In [1]:
# Importing the relevant libraries for the defined problem
import numpy as np
import math
from math import sqrt
import cmath
import scipy
import pandas as pd
import csv
import os
from numpy import linalg
from qutip import *
from numpy.linalg import matrix_rank
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd

In [2]:
def Partial_transposition(M): 
    
    N = np.zeros((4,4), dtype = complex)    
    for i in range(4): 
        for j in range(4): 
            N[i][j] = M[i][j]
    # Putting the partial transpose operation by brute force        
    N[0][1]= M[1][0]               
    N[1][0]= M[0][1]
    N[1][2]= M[0][3]
    N[2][1]= M[3][0]
    N[0][1]= M[1][0]
    N[0][3]= M[1][2]
    N[3][0]= M[2][1]
    N[2][3]= M[3][2]
    N[3][2]= M[2][3]
    
    return N 

def Entangled(M):
    # Function which returns 'True' if Separable and 'False' otherwise    
    Dmat_PT = Partial_transposition(M)
    Dmat_PT_E = np.linalg.eigvalsh(Dmat_PT)
  
    if np.all(Dmat_PT_E > 0) == True :
        # 0 means the state is separable, all it's eigenvalues are non-negative
        return 0  
    else :
        # 1 means the state is entangled, this is of our interest
        return 1  
    
    
# defining functions for checking positive-semi-definiteness 
def PSD(M, tol):
    eigen_val = np.linalg.eigvalsh(M)
    if np.all(eigen_val >= -tol ) == True :
        return True 
    else :
        return False 

def check_valid_rho(M, tol_psd):
    
    shape = M.shape
    eigen_values = np.linalg.eigvalsh(M)
    psd = PSD(M, tol_psd)
    trace = M.trace()
    herm = np.all(M == M.conj().T)
    
    return shape, eigen_values, psd, trace, herm

def check_pure_rho(M, tol_idem):
    
    rank = np.linalg.matrix_rank(M)
    M_square = np.matmul(M,M)
    idempotent = np.all(abs(M_square - M) <= tol_idem )
    #idempotent = M_square == M
    trace = M_square.trace()
    
    return rank, idempotent, trace

def check_rho(M,tol_psd, tol_idem):
    
    '''
    print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},
          \n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))
    '''
    sh, eig, psd, tr, he = check_valid_rho(rho_fs, tol_psd)
    rank, idem, tr =  check_pure_rho(rho_fs, tol_idem)
    return sh, eig, psd, tr, he, rank, idem, tr 

def CNOT_23(M):
    CNOT =    np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    CNOT_d =  np.conjugate(CNOT).T
    Identity = np.eye(2)
    CNOT = Qobj(CNOT)
    CNOT_d = Qobj(CNOT)
    Id = Qobj(Identity)
    U = Qobj(tensor(Id, CNOT).data.toarray().reshape(8,8))
    U_d = U.dag()
    Oper = U*M*U_d
    
    return Oper

def CNOT_12(M):
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    #CNOT_d =  np.conjugate(CNOT).T
    Identity = np.eye(2)
    CNOT = Qobj(CNOT)
    Id = Qobj(Identity)
    U = Qobj(tensor(CNOT,Id).data.toarray().reshape(8,8))
    U_d = U.dag()
    Oper = U*M*U_d
    
    return Oper

def CNOT_13(M):
    # taken from https://quantumcomputing.stackexchange.com/questions/4252/how-to-derive-the-cnot-matrix-for-a-3-qubit-system-where-the-control-target-qu
    CNOT_13   = np.array([[1,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,0],[0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,1],
                       [0,0,0,0,1,0,0,0],[0,2,0,0,0,0,0,0],[0,0,0,0,0,0,1,0],[0,0,0,1,0,0,0,0]])
    CNOT_13   = Qobj(CNOT_13)
    Operation = CNOT_13*M*(CNOT_13.dag())
    return Operation

def CNOT_13_state(M):
    CNOT_13   = np.array([[1,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,0],[0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,1],
                       [0,0,0,0,1,0,0,0],[0,2,0,0,0,0,0,0],[0,0,0,0,0,0,1,0],[0,0,0,1,0,0,0,0]])
    CNOT_13   = Qobj(CNOT_13)
    Operation = CNOT_13*M
    return Operation

def rho(n):
    state_vector = (2*np.random.random(2**n)-1) + (2*np.random.random(2**n)-1)*1j 
    norm = linalg.norm(state_vector)
    state_vector /= norm
    state_vector = Qobj(state_vector)
    rho_ = ket2dm(state_vector)
    return rho_.data.toarray()

def RHO_FS():
    rho_a          = Qobj(rho(1))
    rho_b          = Qobj(rho(1))
    rho_c          = Qobj(rho(1))
    rho_ab         = tensor(rho_a, rho_b)
    rho_abc        = tensor(rho_ab, rho_c)

    return rho_abc.data.toarray()

def rand_bs3():
    rho_bs3 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs3 = CNOT_12(rho_bs3)
    return rho_bs3.data.toarray()

def rand_bs1():
    rho_bs1 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs1 = CNOT_23(rho_bs1)
    return rho_bs1.data.toarray()

def swap_12(M):
    swap = Qobj(np.array([[1,0,0,0],[0,0,1,0], [0,1,0,0], [0,0,0,1]]))
    Identity = Qobj(np.eye(2))
    U = Qobj(tensor(swap, Identity).data.toarray(), dims =  [[8], [8]])
    Ud = U.dag()
    oper = U*M*Ud
    return oper

def rand_bs2():
    rho_bs2 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs2 = CNOT_23(rho_bs2)
    rho_bs2 = swap_12(rho_bs2)
    return rho_bs2.data.toarray()

def GHZ():
    #cnot_12 = CNOT_12(M)
    #Operation = CNOT_13(cnot_12)
    ghz = ghz_state(3).data.toarray()
    ghz[0][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    ghz[7][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    ghz = ghz/np.linalg.norm(ghz)
    ghz = ket2dm(Qobj(ghz, dims = [[2,2,2],[1,1,1]]))
    return ghz.data.toarray()

def W():
    w = w_state(3).data.toarray()
    w[0][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[1][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[2][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[4][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w = w/np.linalg.norm(w)
    w = ket2dm(Qobj(w, dims = [[2,2,2],[1,1,1]]))
    return w.data.toarray()

def array_to_label_list(Matrix,n,k):
    '''
    Matrix refers to the density matrix which is supposed to be input in this function, 
    of whose the list is to be constructed
    
    n = Number of qubits in the system
    
    k = index reference to the three qubit entanglement class 
        k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
        k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
        k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
        k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
        k = 4 => W => Werner State => ABC =>      [0,0,0,0,1,0]
        k = 5 => GHZ => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
        
    From 0 to 31 elements of the returned list real components of the density matrix(Re(rho_00),Re(rho_01),...) i.e, from 0 to N-1
    from 32 to 63 : Imaginary components of the density matrix i.e, (Im(rho_00),Im(rho_01),...) 
    from 64 to 69 : One hot encoded labels corresponding to the class of the 3 qubit entangled states
    '''
    N = 2**(2*n)
    #Matrix   = Matrix.data.toarray()
    Matrix_1 = Matrix.reshape((1,N))
    data = np.zeros((2*N))
    FS  = [1,0,0,0,0,0,0]
    bs2 = [0,1,0,0,0,0,1]
    BS2 = [0,0,1,0,0,0,2]
    BS3 = [0,0,0,1,0,0,3]
    W   = [0,0,0,0,1,0,4]
    GHZ = [0,0,0,0,0,1,5]
    
    for i in range(0,N):
        data[i] = np.real(Matrix_1[0][i])
    for j in range(N,2*N):
        data[j] = np.imag(Matrix_1[0][j-N])
        
    data  = data.tolist()
    if k == 0:
        data.extend(FS)
    elif k == 1:
        data.extend(bs2)
    elif k == 2:
        data.extend(BS2)
    elif k == 3:
        data.extend(BS3)
    elif k == 4:
        data.extend(W)
    elif k == 5:
        data.extend(GHZ)
    else :
        return "Error : Corresponding state label doesn't exist, select from 0,1,2,3,4 or 5"
    return data

def labels(n):
    
    N=2**(2*n)
    fields = [None]*((2*N))
    #list_classes_0 = ['Fully-Separable', 'Bi-Separable-1','Bi-Separable-2','Bi-Separable-3','W','GHZ']
    list_classes_1 = ['FS','BS1','BS2','BS3','W','GHZ','CODE']
     
    k = 0 
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Real'
            k=k+1
            
    k = N
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
            k=k+1
    
    fields.extend(list_classes_1)
    
    return fields

def class_return_3q(k):
    
    if k == 0:
        return RHO_FS()
    elif k == 1:
        return rand_bs1()
    elif k == 2:
        return rand_bs2()
    elif k == 3:
        return rand_bs3()
    elif k == 4:
        return W()
    elif k == 5:
        return GHZ()
    else :
        return "Class index out of [0,1,2,3,4,5]"
    
def data_generator_3q_class(num, k):
    data_list = []
    
    '''
    K = The type of entanglement class one would want to generate with this
            function
    k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
    k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
    k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
    k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
    k = 4 => W => W State => ABC =>      [0,0,0,0,1,0]
    k = 5 => GHZ => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
    Num = No. of quantum states of the class required 
    '''
    
    for i in range(num):
        Matrix = class_return_3q(k)
        data = array_to_label_list(Matrix=Matrix,n=3,k=k)
        data_list.append(data)
    return data_list

In [3]:
'''
Useful set of codes : 

# To delete all the rows in a csv file
f = open(data_0, "w+")
f.close()
   
# To extract the variable name from a variable 

!pip install varname
from varname import nameof
s = 'Hey!'
print (nameof(s))

# 
'''

'\nUseful set of codes : \n\n# To delete all the rows in a csv file\nf = open(data_0, "w+")\n   f.close()\n'

In [ ]:
'''
# Code for metrics to certify the generated quantum states
# for Fully separable and biseparable states we have the rank 
# we have the rank of the reduced density matrices and for the 
# entangled states we have the 3-tangle metric

This space is left for putting the code for the verification 
of the quantum states

It will include function for:
A. implementing the metrics
B. Generating multiple states from the distribution
C. Doing the statistical analysis for the error in generation
for the quantum states. 

'''
'''
AUTOMATED CODE FOR APPENDINNG LIST DATA TO CSV FILES

for i in range(len(list_all)):
    for state in list_all[i]:
         with open(csv_all[i] ,'a') as cf:
                write = csv.writer(cf)   
                write.writerow(state) 
                
start_time = time.time()

for i in range(len(list_all)):
    for state in list_all[i]:
         with open(csv_all[i] ,'a') as cf:
                write = csv.writer(cf)   
                write.writerow(state) 
                
print("--- %s seconds to append 6 entanglement classes data to csv " % (time.time() - start_time))      

start_time = time.time()

for data in list_0:
     with open(data_0 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

#data_0 = os.path.abspath(r"rho_data_3q_fs - Sheet1.csv")
data_0 = r"<LOCAL_DATA_DIR>\rho_data_sim\rho_data_3q_0 - Sheet1.csv"
csv_0 = pd.read_csv(data_0, names = labels(3))

csv_0 = pd.read_csv(data_0, names = labels(3))
csv_0

'''

In [3]:
import time

In [4]:
# FS - 10,000 states 
start_time = time.time()
list_0 = data_generator_3q_class(num=100000, k=0)
data_0 = pd.DataFrame(list_0)
data_0.columns = labels(3)
print("--- %s seconds --- for 100K FS states" % (time.time() - start_time))

--- 347.8964169025421 seconds --- for 100K FS states


In [5]:
# BS1 - 10,000 states
start_time = time.time()
list_1 = data_generator_3q_class(num=100000,k=1)
data_1 = pd.DataFrame(list_1)
data_1.columns = labels(3)
print("--- %s seconds --- for 100K BS1 states" % (time.time() - start_time))

--- 514.4353966712952 seconds --- for 100K BS1 states


In [6]:
# BS2- 10,000 states
start_time = time.time()
list_2 = data_generator_3q_class(num=100000,k=2)
data_2 = pd.DataFrame(list_2)
data_2.columns = labels(3)
print("--- %s seconds --- for 100K BS2 states" % (time.time() - start_time))

--- 599.9118638038635 seconds --- for 100K BS2 states


In [7]:
# BS3- 10,000 states
start_time = time.time()
list_3 = data_generator_3q_class(num=100000, k=3)
data_3 = pd.DataFrame(list_3)
data_3.columns = labels(3)
print("--- %s seconds for 100K BS3 states " % (time.time() - start_time))

--- 491.1495499610901 seconds for 100K BS3 states 


In [8]:
# W states- 10,000 states
start_time = time.time()
list_4 = data_generator_3q_class(num=100000, k=4)
data_4 = pd.DataFrame(list_4)
data_4.columns = labels(3)
print("--- %s seconds for 100K W states " % (time.time() - start_time))

--- 279.3313081264496 seconds for 100K W states 


In [9]:
# GHZ states- 10,000 states
start_time = time.time()
list_5 = data_generator_3q_class(num=100000, k=5)
data_5 = pd.DataFrame(list_5)
data_5.columns = labels(3)
print("--- %s seconds for 10K GHZ states " % (time.time() - start_time))

--- 198.59313869476318 seconds for 10K GHZ states 


In [13]:
list_all = [list_0,list_1,list_2,list_3,list_4,list_5]

In [14]:
csv_all =[data_0,data_1,data_2,data_3,data_4,data_5]

In [15]:
data_all = pd.concat(csv_all, axis=0, ignore_index=True)

In [16]:
from pathlib import Path  
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_0.csv') 
data_0.to_csv(filepath)

In [17]:
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_1.csv') 
data_1.to_csv(filepath)

In [18]:
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_2.csv') 
data_2.to_csv(filepath)

In [19]:
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_3.csv') 
data_3.to_csv(filepath)

In [20]:
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_4.csv') 
data_4.to_csv(filepath)

In [21]:
filepath = Path(r'<LOCAL_DATA_DIR>\TRAINING DATA\data_5.csv') 
data_5.to_csv(filepath)

In [22]:
filepath = Path(r'<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_all.csv') 
data_all.to_csv(filepath)

In [30]:
data_0_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_0_1 - Sheet1.csv"
#csv_0 = pd.read_csv(data_0_1, names = labels(3))

start_time = time.time()

for data in list_0:
     with open(data_0_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 102.34770345687866 seconds to append 10^5 3Q FS states to csv 


In [31]:
data_1_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_1_1 - Sheet1.csv"
#csv_0 = pd.read_csv(data_1_1, names = labels(3))

start_time = time.time()

for data in list_1:
     with open(data_1_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 85.42950963973999 seconds to append 10^5 3Q FS states to csv 


In [32]:
data_2_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_2_1 - Sheet1.csv"
#csv_2 = pd.read_csv(data_2_1, names = labels(3))

start_time = time.time()

for data in list_2:
     with open(data_2_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 84.13776135444641 seconds to append 10^5 3Q FS states to csv 


In [33]:
data_3_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_3_1 - Sheet1.csv"
#csv_3 = pd.read_csv(data_3_1, names = labels(3))

start_time = time.time()

for data in list_3:
     with open(data_3_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 98.21264553070068 seconds to append 10^5 3Q FS states to csv 


In [34]:
data_4_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_4_1 - Sheet1.csv"
#csv_4 = pd.read_csv(data_4_1, names = labels(3))

start_time = time.time()

for data in list_4:
     with open(data_4_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 66.39419865608215 seconds to append 10^5 3Q FS states to csv 


In [35]:
data_5_1 = r"<LOCAL_DATA_DIR>\ANN_PCA_DATASET\ANN\data_5_1 - Sheet1.csv"
#csv_5 = pd.read_csv(data_5_1, names = labels(3))

start_time = time.time()

for data in list_5:
     with open(data_5_1 ,'a') as cf:
        write = csv.writer(cf)   
        write.writerow(data)
        
print("--- %s seconds to append 10^5 3Q FS states to csv " % (time.time() - start_time))

--- 66.73948073387146 seconds to append 10^5 3Q FS states to csv 


## 